# Notebook 03 — scVI Batch Correction and Atlas Integration

**Project:** RetinoblastomaAtlas  
**Input:** `data/processed/03_normalized.h5ad`  
**Output:** `data/processed/04_scvi_integrated.h5ad`, UMAP figures, scib metrics

---

## Scientific Background

GSE168434 and GSE249995 were generated by different research groups using different 10x Genomics kit versions, cDNA capture efficiencies, and sequencing depths. Naively combining them without batch correction would result in cells clustering by dataset origin rather than by biology — making it impossible to jointly annotate cell types or compare intraocular vs. extraocular states.

## Why scVI Over Linear Methods?

Classical batch correction methods (ComBat, Harmony) model technical effects as linear additive or multiplicative corrections to log-normalized expression. scRNA-seq count data is fundamentally **non-Gaussian and non-linear**: it follows a **negative binomial distribution** with gene-specific dispersion parameters.

**scVI** (Single-Cell Variational Inference; Lopez et al. 2018) addresses this by:
1. Modelling raw counts directly through a **negative binomial decoder**, preserving count-level statistical properties
2. Learning a **30-dimensional latent representation** where biological variation is encoded and batch effects are explicit conditioning variables
3. Returning both a **latent embedding** (for clustering/UMAP) and **batch-corrected expression** (for DE analysis)

Correct batch integration is the **foundation** of every downstream analysis:
- Annotation requires intraocular and extraocular cells of the same type to cluster together
- RNA velocity requires cone precursor subclusters from both datasets to be properly co-embedded
- Cell-cell communication requires TME cells from both datasets to be jointly characterized

**References:**  
- Lopez R et al. Deep generative modeling for single-cell transcriptomics. *Nat Methods* 2018. https://doi.org/10.1038/s41592-018-0229-2  
- Luecken MD et al. Benchmarking atlas-level data integration. *Nat Methods* 2022. https://doi.org/10.1038/s41592-021-01336-8

## Parameters

In [ ]:
# --- scVI Architecture ---
N_LATENT   = 30     # Dimensions of latent space
N_LAYERS   = 2      # Encoder/decoder layers
N_HIDDEN   = 128    # Hidden units per layer

# --- Training ---
MAX_EPOCHS = 400    # Maximum training epochs (early stopping will usually stop sooner)
LR         = 1e-3   # Learning rate

# --- Clustering ---
LEIDEN_RESOLUTIONS = [0.3, 0.5, 1.0]  # Multiple resolutions: coarse → fine
N_NEIGHBORS        = 30               # KNN graph for UMAP and Leiden

## Setup

In [ ]:
import warnings
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1
scvi.settings.verbosity = 1

ROOT      = Path("..").resolve()
IN_H5AD   = ROOT / "data" / "processed" / "03_normalized.h5ad"
OUT_H5AD  = ROOT / "data" / "processed" / "04_scvi_integrated.h5ad"
MODEL_DIR = ROOT / "data" / "processed" / "scvi_model"
FIG_DIR   = ROOT / "results" / "figures"
TAB_DIR   = ROOT / "results" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"scVI version: {scvi.__version__}")

## Step 1 — Load Normalized Atlas

In [ ]:
print("Step 1 — Loading normalized atlas")
adata = sc.read_h5ad(IN_H5AD)
print(f"  {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"  HVGs: {adata.var['highly_variable'].sum():,}")

## Step 2 — Pre-Integration UMAP (Batch Effect Diagnosis)

Computing UMAP on the un-corrected PCA space shows how severe the batch effect is. If cells cluster by `dataset` (GSE168434 vs GSE249995) rather than by cell type, batch correction is essential. This diagnostic plot is the baseline against which we evaluate scVI's performance.

In [ ]:
print("Step 2 — Pre-integration UMAP (batch effect diagnosis)")
sc.pp.neighbors(adata, use_rep="X_pca_pre_scvi", n_neighbors=N_NEIGHBORS, n_pcs=30)
sc.tl.umap(adata, min_dist=0.3)
adata.obsm["X_umap_pre_scvi"] = adata.obsm["X_umap"].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata, color="dataset",   ax=axes[0], show=False, frameon=False, size=2, alpha=0.7,
           title="Dataset (before scVI)")
sc.pl.umap(adata, color="sample_id", ax=axes[1], show=False, frameon=False, size=2, alpha=0.7,
           title="Sample ID (before scVI)")
plt.suptitle("UMAP before scVI integration — strong batch effect expected",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / "umap_before_integration.pdf", dpi=200, bbox_inches="tight")
plt.show()
print("  If cells cluster by 'dataset', batch correction is critical.")

## Step 3 — Setup scVI Model

scVI is set up with:
- **`layer='counts'`**: raw integer counts (scVI models count data directly, not log-transformed)
- **`batch_key='dataset'`**: the primary batch variable (GSE168434 vs GSE249995)
- **`categorical_covariate_keys=['sample_id']`**: patient-level effects (within-dataset variation)
- **`continuous_covariate_keys=['pct_counts_mt']`**: residual technical signal from cell quality
- **`gene_likelihood='nb'`**: negative binomial decoder (superior to Poisson for overdispersed count data)
- **`dispersion='gene-batch'`**: separate dispersion parameter per gene per batch

In [ ]:
print("Step 3 — Setting up scVI model")
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    batch_key="dataset",
    categorical_covariate_keys=["sample_id"],
    continuous_covariate_keys=["pct_counts_mt"],
)
model = scvi.model.SCVI(
    adata,
    n_latent=N_LATENT,
    n_layers=N_LAYERS,
    n_hidden=N_HIDDEN,
    gene_likelihood="nb",
    dispersion="gene-batch",
    use_layer_norm="both",
    use_batch_norm="none",
)
print(model)

## Step 4 — Train scVI

The model maximises the **ELBO** (Evidence Lower BOund) — a lower bound on the log likelihood of the observed data. Training typically takes 10–30 minutes for ~100k cells on a GPU, or longer on CPU.

**Early stopping** monitors the validation ELBO with patience=20 epochs to prevent overfitting. The training history plot should show a smoothly decreasing loss that plateaus — not a still-declining loss at max_epochs (under-training) or an oscillating/increasing loss (instability).

In [ ]:
print(f"Step 4 — Training scVI (max_epochs={MAX_EPOCHS}, early stopping patience=20)")
print("  This may take 10-60 minutes depending on hardware (GPU vs CPU).")

model.train(
    max_epochs=MAX_EPOCHS,
    early_stopping=True,
    early_stopping_patience=20,
    early_stopping_monitor="elbo_validation",
    plan_kwargs={"lr": LR, "weight_decay": 1e-6},
    check_val_every_n_epoch=5,
)

# Save trained model
model.save(str(MODEL_DIR), overwrite=True)
print(f"  Model saved → {MODEL_DIR.name}/")

# Training history plot
history = model.history
train_loss = history.get("train_loss_epoch", [])
val_loss   = history.get("elbo_validation",  [])
fig, ax = plt.subplots(figsize=(8, 5))
if len(train_loss):
    ax.plot(train_loss, label="Training ELBO",   color="#4C9BE8")
if len(val_loss):
    ax.plot(val_loss,   label="Validation ELBO", color="#E84C4C", linestyle="--")
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("ELBO loss", fontsize=11)
ax.set_title("scVI training history (lower = better)", fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(FIG_DIR / "scvi_training_history.pdf", dpi=200, bbox_inches="tight")
plt.show()

## Step 5 — Extract Latent Representation and Batch-Corrected Expression

The 30-dimensional latent space encodes biological cell identity with batch effects modelled out. This embedding is used for:
- **KNN neighbour graph** construction for UMAP and Leiden clustering
- **RNA velocity** moments computation (notebook 07)
- **CellRank** kernel construction (notebook 08)

The batch-corrected expression matrix (decoded through a reference batch condition) is stored in `.layers['scvi_normalized']` for downstream differential expression and pathway activity scoring.

In [ ]:
print("Step 5 — Extracting latent representation")
adata.obsm["X_scVI"] = model.get_latent_representation()
print(f"  Latent space: {adata.obsm['X_scVI'].shape} (cells × latent dims)")

print("  Computing batch-corrected normalized expression...")
adata.layers["scvi_normalized"] = model.get_normalized_expression(library_size=1e4)
print("  Stored in .layers['scvi_normalized']")

## Step 6 — KNN Graph, UMAP, and Leiden Clustering

The KNN graph is built in the scVI latent space. Multiple Leiden resolutions are computed:
- **Low resolution (0.3)**: broad cell-type clusters (tumour vs immune vs glial)
- **Medium resolution (0.5)**: recommended for cell-type annotation
- **High resolution (1.0)**: fine subclusters within cone precursors (used for trajectory analysis)

In [ ]:
print("Step 6 — Building KNN graph on scVI latent space")
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=N_NEIGHBORS)
sc.tl.umap(adata, min_dist=0.3)

print(f"  Running Leiden clustering at resolutions: {LEIDEN_RESOLUTIONS}")
for res in LEIDEN_RESOLUTIONS:
    sc.tl.leiden(adata, resolution=res, key_added=f"leiden_{res:.1f}")
    n_clusters = adata.obs[f"leiden_{res:.1f}"].nunique()
    print(f"  Resolution {res}: {n_clusters} clusters")

In [ ]:
# Post-integration UMAPs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata, color="dataset",   ax=axes[0], show=False, frameon=False, size=2, alpha=0.7,
           title="Dataset (after scVI)")
sc.pl.umap(adata, color="sample_id", ax=axes[1], show=False, frameon=False, size=2, alpha=0.7,
           title="Sample ID (after scVI)")
plt.suptitle("UMAP after scVI integration — batch labels should be mixed",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / "umap_after_integration_dataset.pdf", dpi=200, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata, color="leiden_0.5", ax=axes[0], show=False, frameon=False, size=2, alpha=0.7,
           legend_loc="right margin", title="Leiden clusters (res=0.5)")
sc.pl.umap(adata, color="leiden_1.0", ax=axes[1], show=False, frameon=False, size=2, alpha=0.7,
           legend_loc="right margin", title="Leiden clusters (res=1.0)")
plt.suptitle("UMAP after scVI integration — Leiden clusters",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / "umap_after_integration_clusters.pdf", dpi=200, bbox_inches="tight")
plt.show()

## Step 7 — Integration Quality Evaluation (scib-metrics)

Integration quality is evaluated with two competing objectives:
- **Batch correction**: iLISI, kBET, graph connectivity — how well batches are mixed
- **Biological conservation**: cLISI, NMI, ARI — how well biological clusters are preserved

A good integration maximises both, giving an **overall score > 0.7** (Luecken et al. 2022).

In [ ]:
print("Step 7 — Integration quality evaluation (scib-metrics)")
try:
    import scib
    metrics = scib.metrics.metrics(
        adata,
        adata_int=adata,
        batch_key="dataset",
        label_key="leiden_0.5",
        embed="X_scVI",
        cluster_key="leiden_0.5",
        organism="human",
        verbose=False,
    )
    metrics_df = pd.DataFrame(metrics).T
    metrics_df.to_csv(TAB_DIR / "scib_integration_metrics.csv")
    print(f"  scib metrics saved")
    display(metrics_df)
except ImportError:
    print("  scib not installed — skipping integration metrics.")
    print("  Install with: pip install scib-metrics")
    print("  Visually assess: datasets should be mixed in the UMAP above.")

## Step 8 — Save Integrated Atlas

In [ ]:
print(f"Saving integrated atlas → {OUT_H5AD.name}")
adata.write_h5ad(OUT_H5AD, compression="gzip")
size_mb = OUT_H5AD.stat().st_size / 1e6
print(f"  {adata.n_obs:,} cells × {adata.n_vars:,} genes | {size_mb:.0f} MB")
print("\n  AnnData structure:")
print("    .X                      : log-normalized expression")
print("    .layers['counts']       : raw UMI counts")
print("    .layers['scvi_normalized']: batch-corrected expression")
print("    .obsm['X_scVI']         : 30-d scVI latent space")
print("    .obsm['X_umap']         : 2-d UMAP coordinates")
print("    .obs['leiden_*']        : Leiden cluster labels")
print(f"\n  Next: Run notebook 04_cell_type_annotation.ipynb")